In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import hashlib

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/newd.csv')

In [ ]:
print(df.head())

                                              TxHash  BlockHeight  ...  Input Class
0  0xaca3850ba0080cf47b47f80e46da452f61bcbb5470d3...      5848095  ...     0x     0
1  0x95681862f9778e49caecf603dd911d6ed57f7799d89d...      5848181  ...     0x     0
2  0x716ae3961b50186a0bbc272cfcc4555662f7fe33550f...      5848716  ...     0x     0
3  0xf397197b800d6cc055a4db265b5e9df3dd2aa745c813...      5849038  ...     0x     0
4  0x7f8086011a32f128dba57fe06fc5f4a181d2f5401e5a...      5849437  ...     0x     0

[5 rows x 9 columns]


In [ ]:
num_rows = len(df)
print(num_rows)

84665


In [ ]:
print("Class Distribution:")
print(df['Class'].value_counts())

Class Distribution:
Class
0    79216
1     5449
Name: count, dtype: int64


In [ ]:
#BALANCING THE DATASET
majority_class = df[df['Class'] == 0]
minority_class = df[df['Class'] == 1]

In [ ]:
target_minority_size = int(0.4 * len(df))

In [ ]:
minority_oversampled = resample(
    minority_class,
    replace=True,
    n_samples=target_minority_size,
    random_state=42
)

In [ ]:
data_balanced = pd.concat([majority_class, minority_oversampled])

In [ ]:
data_balanced = data_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
print(len(data_balanced))

113082


In [ ]:
data_balanced['From'] = data_balanced['From'].apply(lambda x: int(x, 16) if pd.notnull(x) and isinstance(x, str) and x.startswith('0x') else x)
data_balanced['To'] = data_balanced['To'].apply(lambda x: int(x, 16) if pd.notnull(x) and isinstance(x, str) and x.startswith('0x') else x)
data_balanced['TxHash'] = data_balanced['TxHash'].apply(lambda x: int(hashlib.sha256(x.encode()).hexdigest(), 16) if pd.notnull(x) and isinstance(x, str) else x)


In [ ]:
data_balanced['From'] = data_balanced['From'].apply(lambda x: int(hashlib.sha256(str(x).encode()).hexdigest(), 16) % 10**8 if pd.notnull(x) else x)
data_balanced['To'] = data_balanced['To'].apply(lambda x: int(hashlib.sha256(str(x).encode()).hexdigest(), 16) % 10**8 if pd.notnull(x) else x)
data_balanced['TxHash'] = data_balanced['TxHash'].apply(lambda x: int(hashlib.sha256(str(x).encode()).hexdigest(), 16) % 10**8 if pd.notnull(x) else x)

In [ ]:
data_balanced['Input'] = data_balanced['Input'].apply(lambda x: int(hashlib.sha256(str(x).encode()).hexdigest(), 16) % 10**8 if pd.notnull(x) else 0)
data_balanced['ContractAddress'] = data_balanced['ContractAddress'].apply(lambda x: int(hashlib.sha256(str(x).encode()).hexdigest(), 16) % 10**8 if pd.notnull(x) else 0)

In [ ]:
numerical_features = ['BlockHeight', 'Value', 'From', 'To', 'TxHash']
scaler = MinMaxScaler(feature_range=(0.1, 0.9))
data_balanced[numerical_features] = scaler.fit_transform(data_balanced[numerical_features])

In [ ]:
print("Preprocessed Dataset:")
print(data_balanced.head())

Preprocessed Dataset:
     TxHash  BlockHeight   TimeStamp      From  ...     Value  ContractAddress     Input  Class
0  0.356046     0.758416  1529711511  0.593834  ...  0.100000                0  65085427      0
1  0.405693     0.605054  1518942231  0.195179  ...  0.100005                0  65085427      0
2  0.701071     0.805839  1533013873  0.797271  ...  0.100000                0   7643862      1
3  0.160154     0.808816  1533220978  0.379457  ...  0.100001                0  65085427      0
4  0.380707     0.653837  1522317692  0.148635  ...  0.100047                0  65085427      0

[5 rows x 9 columns]


In [ ]:
data_balanced.to_csv('prepD.csv', index=False)

In [ ]:
X = data_balanced.drop(columns=['Class'])
y = data_balanced['Class']


In [ ]:
print("NaN values before handling:\n", X.isnull().sum()) #Added to diagnose where NaNs may still be
X = X.fillna(0) #Fill NaNs with 0 before feature selection.
print("NaN values after handling:\n", X.isnull().sum()) #Confirm that NaNs have been handled

NaN values before handling:
 TxHash               0
BlockHeight          0
 TimeStamp           0
From                 0
To                 113
Value                0
ContractAddress      0
Input                0
dtype: int64
NaN values after handling:
 TxHash             0
BlockHeight        0
 TimeStamp         0
From               0
To                 0
Value              0
ContractAddress    0
Input              0
dtype: int64


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(score_func=f_classif, k=3)
X_new = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support(indices=True)]
print("Selected Features:", selected_features)

Selected Features: Index(['BlockHeight', ' TimeStamp', 'Input'], dtype='object')


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier,AdaBoostClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, make_scorer
from sklearn.metrics import f1_score
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
pip install xgboost

In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 5.9 MB/s eta 0:00:00


In [ ]:
classifiers = {
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced'),
    'Naive Bayes': GaussianNB(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(class_weight='balanced'),
    'Extra Tree': ExtraTreesClassifier(class_weight='balanced'),
    'XGBoost': xgb.XGBClassifier(scale_pos_weight=1),
    'AdaBoost': AdaBoostClassifier(),
    'CatBoost': CatBoostClassifier(cat_features=None, verbose=0)
}

In [ ]:
# Print Supplied Set Test Results
print("\nSupplied Set Test Results:")
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n{name} Classifier:")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))


Supplied Set Test Results:

Decision Tree Classifier:
Accuracy: 0.9832
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     15844
           1       0.95      1.00      0.97      6773

    accuracy                           0.98     22617
   macro avg       0.97      0.99      0.98     22617
weighted avg       0.98      0.98      0.98     22617


Naive Bayes Classifier:
Accuracy: 0.7079
              precision    recall  f1-score   support

           0       0.87      0.68      0.77     15844
           1       0.51      0.77      0.61      6773

    accuracy                           0.71     22617
   macro avg       0.69      0.73      0.69     22617
weighted avg       0.76      0.71      0.72     22617


K-Nearest Neighbors Classifier:
Accuracy: 0.9526
              precision    recall  f1-score   support

           0       0.99      0.94      0.97     15844
           1       0.88      0.98      0.93      6773

    accuracy      

In [ ]:
!pip install --upgrade scikit-learn

In [ ]:
print("\nCross-Validation Results:")
f1_scorer = make_scorer(f1_score)
for name, clf in classifiers.items():
    scores = cross_val_score(clf, X, y, cv=10, scoring=f1_scorer)
    print(f"\n{name} Classifier:")
    print(f"Mean F1 Score: {scores.mean():.4f}")
    print(f"F1 Score per fold: {scores}")


Cross-Validation Results:

Decision Tree Classifier:
Mean F1 Score: 0.9754
F1 Score per fold: [0.97535668 0.97618016 0.97504688 0.97354041 0.97295742 0.97732197
 0.97648925 0.97620764 0.97565894 0.97508999]

Naive Bayes Classifier:
Mean F1 Score: 0.6028
F1 Score per fold: [0.62051162 0.58278581 0.59990824 0.61215339 0.61951677 0.60707107
 0.59640691 0.59990586 0.59131025 0.59814714]

K-Nearest Neighbors Classifier:
Mean F1 Score: 0.9298
F1 Score per fold: [0.93183731 0.92890068 0.93074017 0.92816292 0.92687665 0.93412329
 0.9301805  0.93395036 0.92141072 0.9319226 ]

Random Forest Classifier:
Mean F1 Score: 0.9910
F1 Score per fold: [0.99077734 0.98960925 0.99265786 0.98976608 0.98976009 0.99266432
 0.99179607 0.99063506 0.99049012 0.99223216]

Extra Tree Classifier:
Mean F1 Score: 0.9947
F1 Score per fold: [0.99456122 0.99309331 0.99484915 0.99339498 0.99412283 0.99617197
 0.9951492  0.99500294 0.99471055 0.99558824]


/usr/local/lib/python3.10/dist-packages/sklearn/utils/_tags.py:354: FutureWarning: The XGBClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


AttributeError: 'super' object has no attribute '__sklearn_tags__'